# Verify `IM1923_SF5-Sucrose_20251106.nwb`

Quick visual sanity check of the sham-feeding conversion: session/subject metadata, raw photometry per region (gACh4h 470 / rDA3m 565 / 405 gACh4h reference), licking, engagement, DLC distance, and peri-event photometry aligned to lick onset and approach.

Run with the `jdb_to_nwb` conda env (has `pynwb` + `ndx_fiber_photometry`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pynwb import NWBHDF5IO

NWB_PATH = "IM1923_SF5-Sucrose_20251106.nwb"
REGIONS = ["mNacSh", "NacCore"]
WL_COLORS = {470: "#2CA02C", 565: "#D62728", 405: "#8FBF8F"}  # green ACh, red DA, desat. 405 reference

io = NWBHDF5IO(NWB_PATH, mode="r")
nwb = io.read()


def tvec(ts):
    """Time vector (seconds) for a TimeSeries, whether it uses rate or explicit timestamps."""
    if ts.timestamps is not None:
        return np.asarray(ts.timestamps[:])
    n = ts.data.shape[0]
    t0 = ts.starting_time if ts.starting_time is not None else 0.0
    return t0 + np.arange(n) / ts.rate


# Raw + filtered + hampel photometry and digital sync all live in nwb.acquisition.
behavior = nwb.processing["behavior"]
dlc = nwb.processing["dlc"]
print(nwb)

## 1. Session / subject metadata + per-side summary table

In [ ]:
s = nwb.subject
print("session_id:        ", nwb.session_id)
print("session_start_time:", nwb.session_start_time)
print("experimenter:      ", nwb.experimenter)
print("institution / lab: ", nwb.institution, "/", nwb.lab)
print("subject:           ", s.subject_id, s.species, s.sex, s.strain, "DOB", s.date_of_birth)
print("\nnotes:\n", nwb.notes)

nwb.processing["session_metadata"]["session_side_metadata"].to_dataframe().T

## 2. Raw photometry — full session, both regions

Downsampled purely for plotting speed. Expect slow photobleaching decay; the 405 nm gACh4h reference should track common motion/bleaching but not signal transients.

In [ ]:
ds = 20  # plot every 20th sample (~4.3 Hz)
fig, axes = plt.subplots(len(REGIONS), 1, figsize=(13, 6), sharex=True)
for ax, region in zip(axes, REGIONS):
    for wl in (470, 565, 405):
        ts = nwb.acquisition[f"raw_{wl}_{region}"]
        t = tvec(ts)[:]
        ax.plot(t / 60, ts.data[:], color=WL_COLORS[wl], lw=0.6, label=f"{wl} nm")
    ax.set_title(f"Raw photometry — {region}")
    ax.set_ylabel("V")
    ax.legend(loc="upper right", ncol=3, fontsize=8)
axes[-1].set_xlabel("time (min)")
fig.tight_layout()

## 3. Licking overview — cumulative licks, lick rate, burst structure

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for region in REGIONS:
    cl = behavior[f"cumulative_licks_{region}"]
    axes[0].plot(tvec(cl) / 60, cl.data[:], label=region)
    lr = behavior[f"lickrate_1s_{region}"]
    axes[1].plot(tvec(lr) / 60, lr.data[:], lw=0.7, label=region)
axes[0].set_ylabel("cumulative licks"); axes[0].legend()
axes[1].set_ylabel("lick rate (licks/min, 1s bins)"); axes[1].set_xlabel("time (min)"); axes[1].legend()
axes[0].set_title("Licking over the session")
fig.tight_layout()

for region in REGIONS:
    bt = behavior[f"burst_table_{region}"].to_dataframe()
    print(f"{region}: {len(bt)} bursts, mean {bt['avg_licks_per_burst'].mean():.0f} licks/burst, "
          f"mean burst dur {bt['lick_burst_duration_ms'].mean()/1000:.1f} s")

## 4. Zoom: photometry + licks + engagement + DLC distance

A 60 s window in the middle of the session. Check that licks, engagement and short head-to-spout distance line up, and that photometry transients occur during bouts.

In [ ]:
region = "mNacSh"
t_start, t_dur = 35 * 60, 60  # seconds

def window(ts, t0, dur):
    t = tvec(ts)
    m = (t >= t0) & (t <= t0 + dur)
    return t[m], np.asarray(ts.data[:])[m]

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)

for wl in (470, 565):
    t, y = window(nwb.acquisition[f"raw_{wl}_{region}"], t_start, t_dur)
    axes[0].plot(t, y, color=WL_COLORS[wl], lw=0.8, label=f"{wl} nm")
axes[0].set_ylabel("V"); axes[0].legend(loc="upper right"); axes[0].set_title(f"{region}: photometry")

t, y = window(behavior[f"lick_binary_{region}"], t_start, t_dur)
axes[1].fill_between(t, 0, np.nan_to_num(y), step="mid", color="k"); axes[1].set_ylabel("lick")

eng_key = [k for k in behavior.data_interfaces if k.startswith("Engagement") and k.endswith(region) and "auto" in k][0]
t, y = window(behavior[eng_key], t_start, t_dur)
axes[2].fill_between(t, 0, y, step="mid", color="tab:orange"); axes[2].set_ylabel("engaged")
axes[2].set_title(eng_key, fontsize=8)

t, y = window(dlc[f"DLC_Distance_head_middle_spout_{region}"], t_start, t_dur)
axes[3].plot(t, y, color="tab:purple", lw=0.8)
axes[3].set_ylabel("head-spout dist (px)"); axes[3].set_xlabel("time (s)")
fig.tight_layout()

## 5. Peri-event photometry — aligned to lick-bout onset

Average raw signal around the first lick of each burst (bout onset). This is the key biological sanity check: dopamine (rDA3m, 565) and ACh (gACh4h, 470) should show structured responses around consummatory onset, while the 405 nm gACh4h reference should stay comparatively flat.

In [ ]:
def bout_onsets(region, min_gap_s=2.0):
    """Lick-event times separated by >= min_gap_s, i.e. bout starts."""
    le = behavior[f"lick_events_{region}"]
    t = np.asarray(le.timestamps[:])
    if t.size == 0:
        return t
    keep = np.concatenate([[True], np.diff(t) >= min_gap_s])
    return t[keep]

def peri_event(ts, event_times, pre=2.0, post=4.0):
    t = tvec(ts); y = np.asarray(ts.data[:])
    fs = ts.rate if ts.rate is not None else 1.0 / np.median(np.diff(t))
    npre, npost = int(pre * fs), int(post * fs)
    lag = (np.arange(-npre, npost)) / fs
    snips = []
    for et in event_times:
        i = int(round((et - t[0]) * fs))
        if i - npre >= 0 and i + npost < len(y):
            seg = y[i - npre:i + npost].astype(float)
            snips.append(seg - np.nanmean(seg[:npre]))  # baseline-subtract pre-window
    return lag, np.array(snips)

fig, axes = plt.subplots(1, len(REGIONS), figsize=(13, 4.5), sharey=True)
for ax, region in zip(axes, REGIONS):
    onsets = bout_onsets(region)
    for wl in (470, 565, 405):
        lag, snips = peri_event(nwb.acquisition[f"raw_{wl}_{region}"], onsets)
        if snips.size == 0:
            continue
        mean, sem = snips.mean(0), snips.std(0) / np.sqrt(len(snips))
        ax.plot(lag, mean, color=WL_COLORS[wl], label=f"{wl} nm")
        ax.fill_between(lag, mean - sem, mean + sem, color=WL_COLORS[wl], alpha=0.2)
    ax.axvline(0, color="k", ls=":", lw=1)
    ax.set_title(f"{region}  (n={len(onsets)} bouts)")
    ax.set_xlabel("time from bout onset (s)")
axes[0].set_ylabel("baseline-subtracted signal (V)"); axes[0].legend()
fig.tight_layout()

## 6. Peri-event photometry — aligned to spout approach

Uses the `approach_events` from the distance-state detection.

In [ ]:
fig, axes = plt.subplots(1, len(REGIONS), figsize=(13, 4.5), sharey=True)
for ax, region in zip(axes, REGIONS):
    appr = behavior[f"approach_events_{region}"]
    at = tvec(appr); av = np.asarray(appr.data[:])
    approach_times = at[av > 0]
    for wl in (470, 565, 405):
        lag, snips = peri_event(nwb.acquisition[f"raw_{wl}_{region}"], approach_times)
        if snips.size == 0:
            continue
        mean, sem = snips.mean(0), snips.std(0) / np.sqrt(len(snips))
        ax.plot(lag, mean, color=WL_COLORS[wl], label=f"{wl} nm")
        ax.fill_between(lag, mean - sem, mean + sem, color=WL_COLORS[wl], alpha=0.2)
    ax.axvline(0, color="k", ls=":", lw=1)
    ax.set_title(f"{region}  (n={len(approach_times)} approaches)")
    ax.set_xlabel("time from approach (s)")
axes[0].set_ylabel("baseline-subtracted signal (V)"); axes[0].legend()
fig.tight_layout()

## 7. Inventory of everything in the file

In [ ]:
print("ACQUISITION (raw + filtered + hampel photometry, digital sync):")
for k, v in nwb.acquisition.items():
    print(f"  {k:28s} {getattr(v.data, 'shape', '?')}")
for mod in ("behavior", "dlc", "session_metadata"):
    print(f"\n{mod.upper()}:")
    for k in nwb.processing[mod].data_interfaces:
        print(f"  {k}")

In [ ]:
# io.close()  # uncomment when done